# Real-World ETL Data Pipeline

## 📌 Project Overview

This project demonstrates an end-to-end ETL (Extract, Transform, Load) data pipeline using a real-world Superstore sales dataset.

The pipeline extracts raw data, validates data quality, transforms and standardizes the dataset, loads the processed data into a SQLite database, and performs business analysis using SQL.

---

## 🛠️ Technologies Used

- Python
- Pandas
- NumPy
- SQLite
- SQL
- Google Colab

---

## 🔄 ETL Workflow

Extract Data
↓
Data Inspection & Validation
↓
Data Transformation
↓
Load into SQLite
↓
SQL Business Analysis
↓
Data Quality Reporting
↓
Export Processed Dataset

---

## 📊 Key Analysis

- Total Sales, Profit, and Orders
- Category-wise Sales and Profit
- Top Products by Sales
- Regional Sales Performance
- Monthly Sales Trends
- Profit Margin Analysis

---

## 🚀 Key Features

- Data quality validation
- Duplicate detection
- Missing-value validation
- Column name standardization
- Date conversion
- Derived business metrics
- SQL-based analysis
- Processed data export

In [4]:
import pandas as pd
import numpy as np
import sqlite3

In [1]:
from google.colab import files

uploaded = files.upload()

Saving Superstore.csv to Superstore.csv


In [6]:
import io

# Get the uploaded file name
file_name = list(uploaded.keys())[0]

# Read the CSV using latin1 encoding
df = pd.read_csv(
    io.BytesIO(uploaded[file_name]),
    encoding="latin1"
)

print("Dataset loaded successfully!")
print("File name:", file_name)

df.head()

Dataset loaded successfully!
File name: Superstore.csv


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [7]:
# Display dataset structure and data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quantity

In [8]:
print("Dataset Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nColumn Names:")
print(df.columns.tolist())

Dataset Shape: (9994, 21)

Missing Values:
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

Duplicate Rows:
0

Column Names:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [9]:
# Create a copy of the original dataset
cleaned_df = df.copy()

print("Original dataset copied successfully!")

Original dataset copied successfully!


In [10]:
# Standardize column names
cleaned_df.columns = (
    cleaned_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(cleaned_df.columns.tolist())

['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'sub-category', 'product_name', 'sales', 'quantity', 'discount', 'profit']


In [11]:
# Count duplicates before removing them
duplicates_before = cleaned_df.duplicated().sum()

# Remove duplicate rows
cleaned_df = cleaned_df.drop_duplicates()

duplicates_after = cleaned_df.duplicated().sum()

print("Duplicate rows before cleaning:", duplicates_before)
print("Duplicate rows after cleaning:", duplicates_after)

Duplicate rows before cleaning: 0
Duplicate rows after cleaning: 0


In [12]:
# Check missing values in each column
missing_values = cleaned_df.isnull().sum()

print("Missing values per column:")
print(missing_values[missing_values > 0])

Missing values per column:
Series([], dtype: int64)


In [13]:
# Find columns that contain the word 'date'
date_columns = [
    column for column in cleaned_df.columns
    if "date" in column.lower()
]

print("Date columns found:", date_columns)

Date columns found: ['order_date', 'ship_date']


In [14]:
# Convert date columns to datetime format

for column in date_columns:
    cleaned_df[column] = pd.to_datetime(
        cleaned_df[column],
        errors="coerce"
    )

print("Date columns converted successfully!")
print(cleaned_df[date_columns].dtypes)

Date columns converted successfully!
order_date    datetime64[ns]
ship_date     datetime64[ns]
dtype: object


In [15]:
# Display summary statistics for numerical columns
cleaned_df.describe()

,row_id,order_date,ship_date,postal_code,sales,quantity,discount,profit
count,9994.000000,9994,9994,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000
mean,4997.500000,2016-04-30 00:07:12.259355648,2016-05-03 23:06:58.571142912,55190.379428,229.858001,3.789574,0.156203,28.656896
min,1.000000,2014-01-03 00:00:00,2014-01-07 00:00:00,1040.000000,0.444000,1.000000,0.000000,-6599.978000
25%,2499.250000,2015-05-23 00:00:00,2015-05-27 00:00:00,23223.000000,17.280000,2.000000,0.000000,1.728750
50%,4997.500000,2016-06-26 00:00:00,2016-06-29 00:00:00,56430.500000,54.490000,3.000000,0.200000,8.666500
75%,7495.750000,2017-05-14 00:00:00,2017-05-18 00:00:00,90008.000000,209.940000,5.000000,0.200000,29.364000
max,9994.000000,2017-12-30 00:00:00,2018-01-05 00:00:00,99301.000000,22638.480000,14.000000,0.800000,8399.976000
std,2885.163629,NaN,NaN,32063.693350,623.245101,2.225110,0.206452,234.260108


In [16]:
# Identify all numerical columns
numeric_columns = cleaned_df.select_dtypes(
    include=["number"]
).columns.tolist()

print("Numeric columns:")
print(numeric_columns)

Numeric columns:
['row_id', 'postal_code', 'sales', 'quantity', 'discount', 'profit']


In [17]:
# Create a profit margin percentage column

cleaned_df["profit_margin"] = (
    cleaned_df["profit"] / cleaned_df["sales"]
) * 100

# Handle cases where sales might be zero
cleaned_df["profit_margin"] = (
    cleaned_df["profit_margin"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print("Derived column created successfully!")

cleaned_df[["sales", "profit", "profit_margin"]].head()

Derived column created successfully!


,sales,profit,profit_margin
0,261.9600,41.9136,16.00
1,731.9400,219.5820,30.00
2,14.6200,6.8714,47.00
3,957.5775,-383.0310,-40.00
4,22.3680,2.5164,11.25


In [18]:
print("Final Dataset Shape:", cleaned_df.shape)

print("\nTotal Missing Values:")
print(cleaned_df.isnull().sum().sum())

print("\nDuplicate Rows:")
print(cleaned_df.duplicated().sum())

print("\nData Types:")
print(cleaned_df.dtypes)

Final Dataset Shape: (9994, 22)

Total Missing Values:
0

Duplicate Rows:
0

Data Types:
row_id                    int64
order_id                 object
order_date       datetime64[ns]
ship_date        datetime64[ns]
ship_mode                object
customer_id              object
customer_name            object
segment                  object
country                  object
city                     object
state                    object
postal_code               int64
region                   object
product_id               object
category                 object
sub-category             object
product_name             object
sales                   float64
quantity                  int64
discount                float64
profit                  float64
profit_margin           float64
dtype: object


In [19]:
# Create a connection to SQLite database
conn = sqlite3.connect("superstore_etl.db")

# Load transformed data into SQL
cleaned_df.to_sql(
    "superstore_sales",
    conn,
    if_exists="replace",
    index=False
)

print("Data successfully loaded into SQLite database!")

Data successfully loaded into SQLite database!


In [20]:
# Check the first 5 records from the SQL database

query = """
SELECT *
FROM superstore_sales
LIMIT 5;
"""

sql_data = pd.read_sql_query(query, conn)

sql_data

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,region,product_id,category,sub-category,product_name,sales,quantity,discount,profit,profit_margin
0,1,CA-2016-152156,2016-11-08 00:00:00,2016-11-11 00:00:00,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,16.00
1,2,CA-2016-152156,2016-11-08 00:00:00,2016-11-11 00:00:00,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,30.00
2,3,CA-2016-138688,2016-06-12 00:00:00,2016-06-16 00:00:00,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,47.00
3,4,US-2015-108966,2015-10-11 00:00:00,2015-10-18 00:00:00,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,-40.00
4,5,US-2015-108966,2015-10-11 00:00:00,2015-10-18 00:00:00,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,11.25


In [23]:
#1
query = """
SELECT
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    COUNT(DISTINCT order_id) AS total_orders
FROM superstore_sales;
"""

business_summary = pd.read_sql_query(query, conn)

business_summary

,total_sales,total_profit,total_orders
0,2297200.86,286397.02,5009


In [24]:
#2
query = """
SELECT
    category,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit
FROM superstore_sales
GROUP BY category
ORDER BY total_sales DESC;
"""

category_analysis = pd.read_sql_query(query, conn)

category_analysis

,category,total_sales,total_profit
0,Technology,836154.03,145454.95
1,Furniture,741999.80,18451.27
2,Office Supplies,719047.03,122490.80


In [25]:
#3
query = """
SELECT
    product_name,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit
FROM superstore_sales
GROUP BY product_name
ORDER BY total_sales DESC
LIMIT 10;
"""

top_products = pd.read_sql_query(query, conn)

top_products

,product_name,total_sales,total_profit
0,Canon imageCLASS 2200 Advanced Copier,61599.82,25199.93
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.38,7753.04
2,Cisco TelePresence System EX90 Videoconferenci...,22638.48,-1811.08
3,HON 5400 Series Task Chairs for Big and Tall,21870.58,0.00
4,GBC DocuBind TL300 Electric Binding System,19823.48,2233.51
5,GBC Ibimaster 500 Manual ProClick Binding System,19024.50,760.98
6,Hewlett Packard LaserJet 3310 Copier,18839.69,6983.88
7,HP Designjet T520 Inkjet Large Format Printer ...,18374.90,4094.98
8,GBC DocuBind P400 Electric Binding System,17965.07,-1878.17
9,High Speed Automatic Electric Letter Opener,17030.31,-262.00


In [26]:
query = """
SELECT
    region,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit
FROM superstore_sales
GROUP BY region
ORDER BY total_sales DESC;
"""

region_analysis = pd.read_sql_query(query, conn)

region_analysis

,region,total_sales,total_profit
0,West,725457.82,108418.45
1,East,678781.24,91522.78
2,Central,501239.89,39706.36
3,South,391721.91,46749.43


In [27]:
query = """
SELECT
    strftime('%Y-%m', order_date) AS month,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit
FROM superstore_sales
GROUP BY month
ORDER BY month;
"""

monthly_analysis = pd.read_sql_query(query, conn)

monthly_analysis

,month,total_sales,total_profit
0,2014-01,14236.90,2450.19
1,2014-02,4519.89,862.31
2,2014-03,55691.01,498.73
3,2014-04,28295.35,3488.84
4,2014-05,23648.29,2738.71
5,2014-06,34595.13,4976.52
6,2014-07,33946.39,-841.48
7,2014-08,27909.47,5318.11
8,2014-09,81777.35,8328.10
9,2014-10,31453.39,3448.26


In [28]:
# Create a final data quality report

final_report = {
    "Total Rows": len(cleaned_df),
    "Total Columns": len(cleaned_df.columns),
    "Missing Values": int(cleaned_df.isnull().sum().sum()),
    "Duplicate Rows": int(cleaned_df.duplicated().sum()),
    "Database Table": "superstore_sales"
}

report_df = pd.DataFrame(
    list(final_report.items()),
    columns=["Metric", "Value"]
)

report_df

,Metric,Value
0,Total Rows,9994
1,Total Columns,22
2,Missing Values,0
3,Duplicate Rows,0
4,Database Table,superstore_sales


In [29]:
# Save the transformed dataset

cleaned_df.to_csv(
    "processed_superstore_data.csv",
    index=False
)

print("Processed dataset saved successfully!")

Processed dataset saved successfully!
